# 02.1 — Modules and imports

[Class index and setup](README.md) · [Next: 02.2](02.2-virtual-environments-and-pip.ipynb)

**Purpose:** Give validation, calculation, state, and input/output clear places to live.

**In class:** 30 minutes. **Independent work:** 15 minutes, included in the index's 210-minute budget. The separate project is additional workload.

## Learning outcomes

- Use ordinary imports from a supplied package.
- Distinguish importing definitions from running a program.
- Move a small calculation into a module and call it explicitly.

## Starting point

You know basic Python and the idea of a Report collecting validated Observations. The complete local example is supplied in this class folder; no earlier notebook or completed exercise is required. Every part imports the names it uses. In VS Code, select this class's `.venv`, then use **Restart Kernel → Run All**.

Default execution only reads local code/data. Terminal installation and file-editing commands are clearly marked for manual use on scratch copies; Run All does not perform them. Empty answer cells are safe. Keep attempts separate from reference cells: uncomment `%load` only after attempting the task, run it to inspect source, then run the populated cell separately. Loading can replace its invoking cell; restarting a kernel does not remove loaded source.

In [ ]:
from pathlib import Path
import sys

if not Path("solutions/exercise_02_1_e1.py").is_file() or not Path("report_app/models.py").is_file():
    raise FileNotFoundError(
        "Open 02-reproducible-python-projects.code-workspace and restart the kernel. "
        "Download the complete class folder, including report_app/ and solutions/."
    )
print("Notebook kernel:", sys.executable)
print("Class package and reference files are available.")

## In class · 0–20 minutes — A project is more than a long script

A **module** is a Python file that can define reusable names. A **package** groups modules under one importable name. The `report_app/` folder is a small package; its `__init__.py` marks the package and deliberately does not run the program.

Open these files in VS Code alongside the notebook:

| File | Responsibility |
| --- | --- |
| [models.py](report_app/models.py) | Observation's input contract |
| [analysis.py](report_app/analysis.py) | Pure summary calculation |
| [report.py](report_app/report.py) | Report state and operations |
| [`__main__.py`](report_app/__main__.py) | Read, validate, and print when explicitly run |
| [sample.json](report_app/sample.json) | Two local input records |

The boundaries explain where to look when something changes. Splitting every function into its own file would add navigation without necessarily adding clarity. Here each file has a concrete role.

In [ ]:
from report_app.models import Observation
from report_app.analysis import summarize
from report_app.report import Report

records = [Observation(label="north", value=2.5), Observation(label="south", value=3.5)]
report = Report("notebook")
for record in records:
    report.add(record)
print(summarize(records))
assert report.summary() == {"count": 2, "total": 6, "mean": 3}

### Illustration — Modular approach code reuse

<img src="images/module-reuse.png" alt="Three mathematical operations reuse a shared helper for finding a hypotenuse." width="750" style="max-width: 100%; height: auto; background-color: white;">

Several operations can reuse one clearly named calculation. The mathematical labels are examples; no trigonometry is required here. In our package, Report uses the summary calculation from analysis.py through an ordinary import. Reuse does not require splitting every function into its own file.

*Credit: [Modular approach code reuse](https://commons.wikimedia.org/wiki/File:Modular_approach_code_reuse.svg) by Kayau, [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). Commons PNG rendering; no content changes. [Image sources and teaching notes](IMAGE_SUGGESTIONS.md).*

### Visual guide — Read the actual package map

<img src="images/package-map.png" alt="The module entrypoint uses models and Report; Report uses models and the summarize function in analysis." width="784" style="max-width: 100%; height: auto; background-color: white;">

Read arrows as imports/uses. Running python -m report_app enters __main__.py; importing report_app.report does not run that entrypoint. Work from the documented class root.

*Credit: Original ADPRO course diagram (2026), [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). [Editable Mermaid source](images/package-map.mmd); [sources and teaching notes](IMAGE_SUGGESTIONS.md).*

### What does the dotted name mean?

`from report_app.analysis import summarize` finds the `analysis.py` module inside `report_app` and binds its `summarize` name in this notebook. Within that package, `.models` is a relative import referring to its own models module. Both are normal imports, not notebook execution commands.

The class workspace makes the class directory the working location. Do not fix an import by appending random paths to `sys.path`. Check the folder you opened, the module filename, and the selected environment first. Avoid naming a file `json.py` or `pydantic.py`, which can hide a library with the same name.

In [ ]:
import report_app.analysis as analysis

print("Loaded module:", analysis.__file__)
print("Callable name:", analysis.summarize.__name__)

### An entry point makes running explicit

The application puts file reading and printing inside `main()`. The guard `if __name__ == "__main__":` runs it when invoked as a program, not when imported for reuse. `python -m report_app` runs the package's `__main__.py` in package context, so its relative imports work.

From the class terminal, run manually:

```sh
uv run --locked python -m report_app
```

The next notebook cell runs only that supplied, local application with the selected Python. It performs no installation or file changes.

In [ ]:
import json
import subprocess

completed = subprocess.run([sys.executable, "-m", "report_app"], capture_output=True, text=True, check=True)
result = json.loads(completed.stdout)
print(result)
assert result == {"name": "class-02", "count": 2, "total": 6, "mean": 3}

### 🧩 Exercise 02.1.E1 — Move a calculation into a module

> **In class · 10 minutes**
>
> **Goal:** Make a small calculation reusable without import-time output.
>
> **Start here:** A new scratch folder in VS Code, the package example, and the answer cell for recording your steps.
>
> **Task:** Put a mean_value(values) function in calculations.py and import it from run_report.py. Return None for empty input. Guard the caller so importing it does not print the report. Run it from the scratch directory.
>
> **Check:** Values 2 and 4 produce 3.0; empty input gives None. The caller imports normally, without editing sys.path.
>
> **Hint:** The calculation needs all inputs explicitly. The caller owns printing; use an `if __name__ == "__main__":` guard.

In [ ]:
# Your attempt for 02.1.E1.
# Record code, commands, observations, and explanation here.

In [ ]:
# Reveal only after attempting the exercise.
# Remove the leading '# ' from the next line and run this cell.
# %load solutions/exercise_02_1_e1.py

## Guided follow-up — Independent work · 15 minutes

Trace one observation through the supplied modules: raw dictionary → Observation → Report → summary function → printed JSON. The import dependencies point toward small reusable components; the input/output entry point uses them rather than the model importing the whole application.

A notebook kernel caches imports. Editing a `.py` file on disk does not guarantee that an already imported module is refreshed. For this course, restart the kernel after editing application modules, then Run All. This is clearer than trying to repair a partially updated graph of imported objects.

The example locates its bundled data relative to `__file__`, the entry module's file. It does not assume every terminal happens to start beside the data. The package still has to be importable: run `python -m report_app` from the documented class/project directory.

Within these 15 minutes, identify the entry point, explain one boundary, and describe what would be surprising if importing a module printed results or started a download. This follow-up is part of the 75-minute refactoring allocation across the block.

**Optional extension · 15 minutes, excluded:** try a small command-line argument for the report name. Package publication and build backends are not required.

## References

- [Python modules](https://docs.python.org/3/tutorial/modules.html)
- [Next: environments and pip](02.2-virtual-environments-and-pip.ipynb)
- [Class index](README.md)